In [36]:
!pip install scikit-learn==1.8.0 lightgbm catboost

In [1]:
from google.colab import drive
drive.mount('/content/drive')

%cd "/content/drive/MyDrive/Colab Notebooks/LG Aimers"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Colab Notebooks/LG Aimers


In [2]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.isotonic import IsotonicRegression
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')

In [3]:
# =========================================================
# 앙상블 가중치 설정 (안정적인 5:5 앙상블)
# =========================================================
W_LGB = 0.5
W_CAT = 0.5

In [4]:
# =========================================================
# Custom Class Definition
# =========================================================
class EnsembleCalibratedPipeline:
    def __init__(self, lgb_pipeline, cat_pipeline, calibrator, w_lgb=0.5, w_cat=0.5):
        self.lgb_pipeline = lgb_pipeline
        self.cat_pipeline = cat_pipeline
        self.calibrator = calibrator
        self.w_lgb = w_lgb
        self.w_cat = w_cat

    def predict_proba(self, X):
        lgb_probs = self.lgb_pipeline.predict_proba(X)[:, 1]
        cat_probs = self.cat_pipeline.predict_proba(X)[:, 1]

        raw_ensemble = self.w_lgb * lgb_probs + self.w_cat * cat_probs
        calibrated_probs = self.calibrator.predict(raw_ensemble)
        return np.column_stack([1 - calibrated_probs, calibrated_probs])

In [10]:
# =========================================================
# Feature Engineering 유틸리티 함수 (Bayesian Smoothing 추가)
# =========================================================
def engineer_features(df, global_mean=0.5521, m=30):
    """
    df: 입력 데이터프레임
    global_mean: 리그 전체 평균 제구 성공률 (Train 기준 약 0.5521)
    m: 베이지안 스무딩 가중치 파라미터 (표본 수가 m개일 때 리그 평균과 50:50 가중)
    """
    df = df.copy()

    # ---------------------------------------------------------
    # 🔥 [신규 추가] Bayesian Smoothing 적용
    # ---------------------------------------------------------
    # asof_pitcher_n이 없거나 NaN인 경우 0으로 대체, success_rate는 global_mean으로 대체
    n = df['asof_pitcher_n'].fillna(0)
    rate = df['asof_pitcher_success_rate'].fillna(global_mean)

    # Smoothed Rate = (n * rate + m * global_mean) / (n + m)
    df['smoothed_pitcher_success_rate'] = (n * rate + m * global_mean) / (n + m)

    # ---------------------------------------------------------
    # 기존 파생 피처들
    # ---------------------------------------------------------
    # 1. is_home: 투수 기준 홈경기 1, 원정 0
    df['is_home'] = (df['score_diff_home'] == df['score_diff_pitcher_team']).astype(int)

    # 2. count_advantage: 볼카운트 유불리 (스트라이크 수 - 볼 수)
    df['count_advantage'] = df['strikes_before'] - df['balls_before']

    # 3. must_strike: 3볼 상황에서 스트라이크가 2개 미만인 압박 상황
    df['must_strike'] = ((df['balls_before'] == 3) & (df['strikes_before'] < 2)).astype(int)

    # 4. is_full_count: 3볼 2스트라이크 풀카운트 상황
    df['is_full_count'] = ((df['balls_before'] == 3) & (df['strikes_before'] == 2)).astype(int)

    # 5. run_pitcher_before / run_batter_before
    is_top = (df['top_bottom'] == 'T')
    df['run_pitcher_before'] = np.where(is_top, df['run_bot_before'], df['run_top_before'])
    df['run_batter_before'] = np.where(is_top, df['run_top_before'], df['run_bot_before'])

    # 6. is_scoring_position: 득점권 유무
    df['is_scoring_position'] = ((df['runner_on_2b'] == 1) | (df['runner_on_3b'] == 1)).astype(int)

    # 7. is_bases_loaded: 만루 상황
    df['is_bases_loaded'] = ((df['runner_on_1b'] == 1) & (df['runner_on_2b'] == 1) & (df['runner_on_3b'] == 1)).astype(int)

    # 8. pitcher_win_expectancy: 투수팀 기준 승리 기대 확률
    df['pitcher_win_expectancy'] = np.where(df['is_home'] == 1, df['home_win_expectancy'], df['away_win_expectancy'])

    # 9. platoon_advantage: 투타 손 상성 (같으면 투수 유리 1, 다르면 0)
    df['platoon_advantage'] = (df['pitcher_hand'] == df['batter_hand']).astype(int)

    # 10. recent_control_momentum: 최근 1경기 폼 모멘텀
    df['recent_control_momentum'] = df['asof_pitcher_prev1_game_success_rate'] - df['asof_pitcher_success_rate']

    # ---------------------------------------------------------
    # 자유로운 피처 제거 (DROP_COLS)
    # ---------------------------------------------------------
    DROP_COLS = [
        # "season",
        "game_dayofweek",
        # "inning",
        # "top_bottom",
        # "game_type",
        # "run_top_before",
        # "run_bot_before",
        # "score_diff_home",
        # "base_state",
        # "batter_hand",
        # "pitcher_team_id",
        # "batter_team_id"
    ]

    cols_to_drop = [c for c in DROP_COLS if c in df.columns]
    df = df.drop(columns=cols_to_drop)

    return df

In [11]:
# =========================================================
# 메인 데이터 로드 및 환경 설정
# =========================================================
DATA_DIR = "./data"
ID = "row_id"
TARGET = "control_success"

# 범주형 피처 지정
CAT_COLS = ["top_bottom", "pitcher_id", "batter_id", "pitcher_team_id", "batter_team_id", "base_state", "game_type"]

train_raw = pd.read_csv(os.path.join(DATA_DIR, "train.csv"), encoding="utf-8-sig")

seasons = train_raw["season"].copy()
y = train_raw[TARGET].copy()

# Global Mean 계산 (Data Leakage 방지를 위해 Train 데이터 전체 평균값 기준)
GLOBAL_MEAN = y.mean()
print(f"📊 Overall Target Mean (GLOBAL_MEAN): {GLOBAL_MEAN:.4f}")

print("🔨 Feature Engineering (Bayesian Smoothing 포함) 적용 중...")
X_engineered = engineer_features(train_raw.drop(columns=[ID, TARGET]), global_mean=GLOBAL_MEAN, m=30)

FEATURES = list(X_engineered.columns)
NUM_COLS = [c for c in FEATURES if c not in CAT_COLS]

print("train:", X_engineered.shape, "| 총 피처 수:", len(FEATURES),
      f"(범주형 {len(CAT_COLS)}, 수치형 {len(NUM_COLS)})")

# Out-of-Time Split (2019~2023: Train / 2024: Validation)
is_val = (seasons == 2024)
X_tr, y_tr = X_engineered.loc[~is_val], y.loc[~is_val]
X_va, y_va = X_engineered.loc[is_val], y.loc[is_val]

print(f"Train (2019~2023): {X_tr.shape} | Val (2024): {X_va.shape}")

preprocessor = ColumnTransformer([
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), CAT_COLS),
    ("num", SimpleImputer(strategy="median"), NUM_COLS),
])

📊 Overall Target Mean (GLOBAL_MEAN): 0.5238
🔨 Feature Engineering (Bayesian Smoothing 포함) 적용 중...
train: (1475092, 58) | 총 피처 수: 58 (범주형 7, 수치형 51)
Train (2019~2023): (1221585, 58) | Val (2024): (253507, 58)


In [12]:
# ---------------------------------------------------------
# 모델 정의 (안정적인 베이스라인 하이퍼파라미터 복구)
# ---------------------------------------------------------
lgbm = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=6,
    num_leaves=31,
    # min_child_samples=30,     # 과적합 방지용 노드 최저 데이터 수
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

cat = CatBoostClassifier(
    iterations=400,
    learning_rate=0.04,
    depth=6,
    # l2_leaf_reg=3.0,
    random_seed=42,
    thread_count=-1,
    verbose=100
)

pipe_lgb = Pipeline([("pre", preprocessor), ("clf", lgbm)])
pipe_cat = Pipeline([("pre", preprocessor), ("clf", cat)])

In [13]:
# ---------------------------------------------------------
# Out-of-Time Validation 및 확률 보정
# ---------------------------------------------------------
print("\n--- 모델 학습 시작 (2019~2023 데이터) ---")
t0 = time.time()
pipe_lgb.fit(X_tr, y_tr)
pipe_cat.fit(X_tr, y_tr)
print(f"학습 완료 :: {time.time() - t0:.1f}s")

# 2024 시즌 검증 예측
val_pred_lgb = pipe_lgb.predict_proba(X_va)[:, 1]
val_pred_cat = pipe_cat.predict_proba(X_va)[:, 1]

raw_val_ensemble = W_LGB * val_pred_lgb + W_CAT * val_pred_cat

# Calibration (2024년 예측값에 보정기 적응)
iso_reg = IsotonicRegression(out_of_bounds='clip')
calibrated_val_preds = iso_reg.fit_transform(raw_val_ensemble, y_va)

# BSS 계산 함수
def calc_bss(y_true, y_pred):
    r = y_true.mean()
    brier = ((y_pred - y_true) ** 2).mean()
    base_brier = r * (1 - r)
    return max(0, 100000 * (1 - brier / base_brier))

print(f"\n📊 [2024 Validation BSS Score]")
print(f" LightGBM 단독 BSS: {calc_bss(y_va, val_pred_lgb):.2f}")
print(f" CatBoost 단독 BSS: {calc_bss(y_va, val_pred_cat):.2f}")
print(f" Raw 앙상블 BSS ({W_LGB}:{W_CAT}): {calc_bss(y_va, raw_val_ensemble):.2f}")
print(f"🔥 보정 후 최종 앙상블 BSS: {calc_bss(y_va, calibrated_val_preds):.2f}")


--- 모델 학습 시작 (2019~2023 데이터) ---
[LightGBM] [Info] Number of positive: 649372, number of negative: 572213
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.199355 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7192
[LightGBM] [Info] Number of data points in the train set: 1221585, number of used features: 58
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.531582 -> initscore=0.126494
[LightGBM] [Info] Start training from score 0.126494
0:	learn: 0.6923352	total: 284ms	remaining: 1m 53s
100:	learn: 0.6823401	total: 30.4s	remaining: 1m 29s
200:	learn: 0.6815184	total: 1m	remaining: 59.5s
300:	learn: 0.6809890	total: 1m 32s	remaining: 30.4s
399:	learn: 0.6804525	total: 2m 1s	remaining: 0us
학습 완료 :: 195.5s

📊 [2024 Validation BSS Score]
 LightGBM 단독 BSS: 683.49
 CatBoost 단독 BSS: 736.48
 Raw 앙상블 BSS (0.5:0.5): 735.88
🔥 보정 후 최종 앙상블 BSS: 811

In [14]:
# ---------------------------------------------------------
# 전체 데이터(2019~2024) 재학습 및 최종 저장
# ---------------------------------------------------------
print("\n--- 전체 데이터(2019~2024) 재학습 및 저장 ---")
t0 = time.time()

pipe_lgb_full = Pipeline([("pre", preprocessor), ("clf", lgbm)])
pipe_cat_full = Pipeline([("pre", preprocessor), ("clf", cat)])

pipe_lgb_full.fit(X_engineered, y)
pipe_cat_full.fit(X_engineered, y)

final_model = EnsembleCalibratedPipeline(
    pipe_lgb_full,
    pipe_cat_full,
    iso_reg,
    w_lgb=W_LGB,
    w_cat=W_CAT
)

print(f"전체 재학습 완료 :: {time.time() - t0:.1f}s")

os.makedirs("./model", exist_ok=True)
joblib.dump(final_model, "./model/feature_engineering4.pkl", compress=3)
print("저장 완료: ./model/feature_engineering4.pkl")


--- 전체 데이터(2019~2024) 재학습 및 저장 ---
[LightGBM] [Info] Number of positive: 772603, number of negative: 702489
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.409693 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7207
[LightGBM] [Info] Number of data points in the train set: 1475092, number of used features: 58
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.523766 -> initscore=0.095136
[LightGBM] [Info] Start training from score 0.095136
0:	learn: 0.6923198	total: 435ms	remaining: 2m 53s
100:	learn: 0.6835373	total: 36.4s	remaining: 1m 47s
200:	learn: 0.6827792	total: 1m 11s	remaining: 1m 10s
300:	learn: 0.6822877	total: 1m 47s	remaining: 35.4s
399:	learn: 0.6818477	total: 2m 23s	remaining: 0us
전체 재학습 완료 :: 234.1s
저장 완료: ./model/feature_engineering4.pkl


In [15]:
# =========================================================
# Feature Importance (피처 중요도) 출력 및 시각화
# =========================================================
transformed_features = CAT_COLS + NUM_COLS

# 1) LightGBM 피처 중요도 추출 (Gain 기준)
lgb_clf = pipe_lgb_full.named_steps['clf']
lgb_importance = lgb_clf.booster_.feature_importance(importance_type='gain')

fi_lgb = pd.DataFrame({
    'feature': transformed_features,
    'lgb_importance_gain': lgb_importance
}).sort_values(by='lgb_importance_gain', ascending=False).reset_index(drop=True)

# 2) CatBoost 피처 중요도 추출
cat_clf = pipe_cat_full.named_steps['clf']
cat_importance = cat_clf.get_feature_importance()

fi_cat = pd.DataFrame({
    'feature': transformed_features,
    'cat_importance': cat_importance
}).sort_values(by='cat_importance', ascending=False).reset_index(drop=True)

# 3) 결과 출력 (상위 15개 & 하위 10개)
print("="*60)
print("🔥 [LightGBM] 상위 15개 핵심 피처 (Gain 기준):")
print(fi_lgb.head(15).to_string(index=False))

print("\n🧊 [LightGBM] 하위 10개 피처 (Gain 기준):")
print(fi_lgb.tail(10).sort_values(by='lgb_importance_gain', ascending=True).to_string(index=False))

print("\n" + "="*60)
print("🔥 [CatBoost] 상위 15개 핵심 피처:")
print(fi_cat.head(15).to_string(index=False))

print("\n🧊 [CatBoost] 하위 10개 피처:")
print(fi_cat.tail(10).sort_values(by='cat_importance', ascending=True).to_string(index=False))
print("="*60)

# 4) 중요도가 0인 피처 확인
zero_lgb = fi_lgb[fi_lgb['lgb_importance_gain'] == 0]['feature'].tolist()
zero_cat = fi_cat[fi_cat['cat_importance'] == 0]['feature'].tolist()

print("\n⚠️ LightGBM에서 중요도 0인 피처:", zero_lgb if zero_lgb else "없음")
print("⚠️ CatBoost에서 중요도 0인 피처:", zero_cat if zero_cat else "없음")

🔥 [LightGBM] 상위 15개 핵심 피처 (Gain 기준):
                             feature  lgb_importance_gain
                              season        101408.747612
           asof_pitcher_success_rate         91911.028250
       smoothed_pitcher_success_rate         56295.983677
                           game_type         51784.723253
           asof_pitcher_reverse_rate         29740.882613
            asof_batter_success_rate         24931.987790
asof_pitcher_prev5_game_success_rate         20026.676889
              asof_pitcher_ball_rate         13090.721897
asof_pitcher_prev1_game_success_rate         12569.333688
                   platoon_advantage         12109.529658
asof_pitcher_prev3_game_success_rate         10079.453920
          asof_pitcher_offspeed_rate          8519.048385
                      asof_pitcher_n          8508.609390
                          pitcher_id          7945.616021
                      batter_team_id          7944.582420

🧊 [LightGBM] 하위 10개 피처 (Gain 기준):
